In [ ]:
from google.colab import files
uploaded = files.upload()

Saving AI_switching_dataset_NEW.csv to AI_switching_dataset_NEW.csv


In [ ]:
"""
=============================================================================
  WiFi 6 + VLC — PyTorch Dueling Double DQN  (All Issues Fixed)
  Dataset : AI_switching_dataset_NEW.csv  (20 500 rows × 15 columns)
=============================================================================
  Labels  :  1=VLC  |  2=WiFi  |  3=MRC
  Actions :  0=VLC  |  1=WiFi  |  2=MRC

  ALL BUGS FIXED
  ──────────────────────────────────────────────────────────────
  FIX 1  Inconsistent confusion matrices
         All plots now use the SAME ev DataFrame from evaluate().
         Dashboard panel is built from that ev — no separate recomputation.

  FIX 2  BER vs SNR differs between fig1 and dashboard
         Dashboard [0,0] now has identical 4 lines (VLC/WiFi/MRC/DQN).

  FIX 3  Training dip at last episode
         Best checkpoint is loaded AFTER the training loop ends.
         The episode that triggers the load is NOT included in logs.

  FIX 4  Wrong action distribution (VLC under-selected)
         W_EXPERT raised to 5.0 so the correct-label bonus always
         dominates the quality term gap → DQN learns to follow labels.
         Added behavioral-cloning pre-training (20 epochs) to warm-start
         Q-values before RL, so VLC is correctly preferred at high SNR.

  FIX 5  Q-value heatmap removed (as requested).

  FIX 6  Confusion matrix is now a single clean figure
         showing both count and row-% annotations in one panel.

  DQN Stack
  ──────────────────────────────────────────────────────────────
  • Dueling DQN          V(s) + A(s,a) − mean A(s,a)
  • Double DQN           online selects, target evaluates
  • Prioritised Replay   α=0.6, β annealed 0.4 → 1.0
  • Huber loss (δ=1)
  • Gradient clipping    L2 ≤ 10
  • Soft target update   τ=0.005 Polyak
  • Cosine ε-schedule    1.0 → 0.01
  • BC pre-training      20 epochs supervised warm-start
  • AdamW + cosine LR    per-episode stepping

  Figures  (9 figures, Q-heatmap removed)
  ──────────────────────────────────────────────────────────────
  fig0  – 3×3 Summary dashboard   (all panels from same ev)
  fig1  – BER vs SNR
  fig2  – Throughput vs SNR
  fig3  – Latency vs SNR
  fig4  – Action distribution per SNR (stacked bar)
  fig5  – Training reward curve  +  ε schedule
  fig6  – Training accuracy  +  |TD| error
  fig7  – Confusion matrix  (single panel, count + row %)
  fig8  – Reward component breakdown per SNR
=============================================================================
"""

import math, os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ─────────────────────────────────────────────────────────────
#  0.  CONFIG
# ─────────────────────────────────────────────────────────────
CSV_PATH  = "AI_switching_dataset_NEW.csv"
SAVE_DIR  = "."
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FEATURE_COLS = [
    "SNR_dB",
    "Mean_hVLC",        "Var_hVLC",
    "Mean_HWiFi",       "Var_HWiFi",
    "Reliability_VLC",  "Reliability_WiFi",
    "BER_VLC",          "BER_WiFi",   "BER_MRC",
    "Latency_VLC_ms",   "Latency_WiFi_ms",
    "Throughput_VLC_Mbps", "Throughput_WiFi_Mbps",
]
N_FEATURES   = len(FEATURE_COLS)    # 14
N_ACTIONS    = 3
LABEL_TO_ACT = {1: 0, 2: 1, 3: 2}  # VLC→0, WiFi→1, MRC→2
ACTION_NAMES = ["VLC", "WiFi", "MRC"]
COLORS = {
    "VLC":  "#0072BD", "WiFi": "#D95319",
    "MRC":  "#77AC30", "DQN":  "#7E2F8E", "grid": "#CCCCCC",
}

# Reward weights
MAX_TPUT    = 311.0
MAX_REL     = 2000.0
W_BER       = 0.8      # kept moderate so expert signal dominates
W_TPUT_VLC  = 3.0
W_TPUT_WIFI = 0.5
W_TPUT_MRC  = 1.5
W_REL       = 0.3
W_EXPERT    = 5.0      # FIX 4: raised from 2 → 5, always beats quality gap
W_SWITCH    = 0.10
REWARD_CLIP = 20.0

# Training
N_EPISODES   = 300
EPISODE_LEN  = 500
GAMMA        = 0.97
LR           = 3e-4
WEIGHT_DECAY = 1e-5
EPS_START    = 1.0
EPS_END      = 0.01
BATCH_SIZE   = 256
BUF_CAPACITY = 50_000
WARM_START   = 512
TAU          = 0.005
HARD_UPD_EP  = 20
GRAD_CLIP    = 10.0
PER_ALPHA    = 0.6
PER_BETA_0   = 0.4
BC_EPOCHS    = 20      # FIX 4: behavioral-cloning warm-start epochs

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


# ─────────────────────────────────────────────────────────────
#  1.  DATA
# ─────────────────────────────────────────────────────────────
def load_dataset(path):
    df     = pd.read_csv(path)
    X_raw  = df[FEATURE_COLS].values.astype(np.float32)
    y      = np.array([LABEL_TO_ACT[l] for l in df["Label"].values], dtype=np.int64)
    scaler = StandardScaler()
    X_scl  = scaler.fit_transform(X_raw).astype(np.float32)

    print("=" * 62)
    print(f"  Dataset   : {path}")
    print(f"  Rows      : {len(df):,}   Features : {N_FEATURES}")
    snr = df["SNR_dB"]
    print(f"  SNR range : {snr.min():.0f}–{snr.max():.0f} dB "
          f"({snr.nunique()} unique, {len(df)//snr.nunique()} rows each)")
    for i, name in enumerate(ACTION_NAMES):
        n = int((y == i).sum())
        print(f"  Action {i} ({name:4s}): {n:6,}  ({n/len(df)*100:.1f}%)")
    print(f"  Device    : {DEVICE}")
    print("=" * 62)
    return X_scl, X_raw, y, scaler, df


# ─────────────────────────────────────────────────────────────
#  2.  REWARD  (FIX 4: W_EXPERT=5.0 dominates quality gap)
# ─────────────────────────────────────────────────────────────
def compute_reward(row, action, optimal, prev_action):
    ber_v = max(float(row["BER_VLC"]),  1e-9)
    ber_w = max(float(row["BER_WiFi"]), 1e-9)
    ber_m = max(float(row["BER_MRC"]),  1e-9)
    tp_v  = float(row["Throughput_VLC_Mbps"])
    tp_w  = float(row["Throughput_WiFi_Mbps"])
    rel_v = float(row["Reliability_VLC"])
    rel_w = float(row["Reliability_WiFi"])
    lat_v = max(float(row["Latency_VLC_ms"]),  1e-6)
    lat_w = max(float(row["Latency_WiFi_ms"]), 1e-6)

    if action == 0:    # VLC
        r_qual = (W_BER * (-math.log10(ber_v))
                  + W_TPUT_VLC  * (tp_v / MAX_TPUT)
                  + W_REL       * (rel_v / MAX_REL))
        r_lat = 0.05 / lat_v
        ber_sel, tput_sel, lat_sel = ber_v, tp_v, lat_v
    elif action == 1:  # WiFi
        r_qual = (W_BER * (-math.log10(ber_w))
                  + W_TPUT_WIFI * (tp_w / MAX_TPUT)
                  + W_REL       * (rel_w / MAX_REL))
        r_lat = 0.05 / lat_w
        ber_sel, tput_sel, lat_sel = ber_w, tp_w, lat_w
    else:              # MRC
        r_qual = (W_BER * (-math.log10(ber_m))
                  + W_TPUT_MRC  * (max(tp_v, tp_w) / MAX_TPUT)
                  + W_REL       * (max(rel_v, rel_w) / MAX_REL))
        r_lat = 0.05 / min(lat_v, lat_w)
        ber_sel = ber_m; tput_sel = max(tp_v, tp_w); lat_sel = min(lat_v, lat_w)

    r_expert = W_EXPERT * (1.0 if action == optimal else -1.0)   # FIX 4
    r_switch = -W_SWITCH * float(action != prev_action)
    reward   = float(np.clip(r_qual + r_lat + r_expert + r_switch,
                             -REWARD_CLIP, REWARD_CLIP))
    return reward, ber_sel, tput_sel, lat_sel, r_qual, r_lat, r_expert, r_switch


# ─────────────────────────────────────────────────────────────
#  3.  ENVIRONMENT
# ─────────────────────────────────────────────────────────────
class LinkEnv:
    def __init__(self, X_scl, y, df, ep_len=EPISODE_LEN):
        self.X_all = X_scl; self.y_all = y
        self.df_all = df.reset_index(drop=True)
        self.N = len(X_scl); self.ep_len = ep_len
        self.X = X_scl; self.y = y
        self.df = df.reset_index(drop=True)
        self.ptr = 0; self.prev = 0

    def reset(self):
        idx = np.random.choice(self.N, size=self.ep_len, replace=True)
        self.X  = self.X_all[idx]
        self.y  = self.y_all[idx]
        self.df = self.df_all.iloc[idx].reset_index(drop=True)
        self.ptr = 0; self.prev = 0
        return self.X[0].copy()

    def step(self, action):
        row = self.df.iloc[self.ptr]
        opt = int(self.y[self.ptr])
        (reward, ber_sel, tput_sel, lat_sel,
         r_qual, r_lat, r_expert, r_switch) = compute_reward(row, action, opt, self.prev)
        info = dict(snr=float(row["SNR_dB"]), ber_sel=ber_sel,
                    ber_vlc=max(float(row["BER_VLC"]),1e-9),
                    ber_wifi=max(float(row["BER_WiFi"]),1e-9),
                    ber_mrc=max(float(row["BER_MRC"]),1e-9),
                    tput_sel=tput_sel, lat_sel=lat_sel, optimal=opt,
                    r_qual=r_qual, r_lat=r_lat,
                    r_expert=r_expert, r_switch=r_switch)
        self.prev = action; self.ptr += 1
        done  = (self.ptr >= self.ep_len)
        next_s = self.X[min(self.ptr, self.ep_len-1)].copy()
        return next_s, reward, done, info


# ─────────────────────────────────────────────────────────────
#  4.  DUELING DQN
# ─────────────────────────────────────────────────────────────
class DuelingDQN(nn.Module):
    def __init__(self, n_in=N_FEATURES, n_out=N_ACTIONS):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(n_in, 512), nn.SiLU(), nn.BatchNorm1d(512), nn.Dropout(0.1),
            nn.Linear(512,  512), nn.SiLU(), nn.BatchNorm1d(512), nn.Dropout(0.1),
            nn.Linear(512,  256), nn.SiLU(), nn.BatchNorm1d(256),
        )
        self.value     = nn.Sequential(nn.Linear(256,128), nn.SiLU(), nn.Linear(128,1))
        self.advantage = nn.Sequential(nn.Linear(256,128), nn.SiLU(), nn.Linear(128,n_out))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.backbone(x)
        V = self.value(f); A = self.advantage(f)
        return V + A - A.mean(dim=1, keepdim=True)

    def act_greedy(self, state):
        self.eval()
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            q = self(s).squeeze(0).cpu().numpy()
        return int(np.argmax(q)), q

    def act_epsilon_greedy(self, state, eps):
        if random.random() < eps:
            self.eval()
            with torch.no_grad():
                s = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                q = self(s).squeeze(0).cpu().numpy()
            return random.randrange(N_ACTIONS), q
        return self.act_greedy(state)


# ─────────────────────────────────────────────────────────────
#  5.  BEHAVIORAL CLONING PRE-TRAINING  (FIX 4)
# ─────────────────────────────────────────────────────────────
def pretrain_bc(net, X_scl, y, epochs=BC_EPOCHS, lr=1e-3):
    """
    Supervised warm-start: train online net to predict the label
    (optimal action) directly before RL starts.
    This ensures Q-values are label-aligned from episode 1.
    """
    print(f"\n  Behavioral cloning pre-training ({epochs} epochs) …")
    net.train()
    opt = optim.Adam(net.parameters(), lr=lr)
    X_t = torch.tensor(X_scl, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y,     dtype=torch.long,    device=DEVICE)
    ds  = torch.utils.data.TensorDataset(X_t, y_t)
    dl  = torch.utils.data.DataLoader(ds, batch_size=512, shuffle=True)

    for ep in range(epochs):
        total_loss = correct = n = 0
        for xb, yb in dl:
            net.train()
            q = net(xb)                          # (B, N_ACTIONS)
            loss = F.cross_entropy(q, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            opt.step()
            total_loss += loss.item() * len(xb)
            correct    += (q.argmax(1) == yb).sum().item()
            n          += len(xb)
        if ep % 5 == 0 or ep == epochs-1:
            print(f"    BC ep {ep+1:3d}/{epochs}  "
                  f"loss={total_loss/n:.4f}  acc={correct/n*100:.1f}%")
    print()


# ─────────────────────────────────────────────────────────────
#  6.  PRIORITISED REPLAY BUFFER
# ─────────────────────────────────────────────────────────────
class PERBuffer:
    def __init__(self, cap, n_s, alpha=PER_ALPHA, beta=PER_BETA_0):
        self.cap, self.alpha, self.beta = cap, alpha, beta
        self.ptr = self.size = 0
        self.S  = np.zeros((cap, n_s), dtype=np.float32)
        self.Sn = np.zeros((cap, n_s), dtype=np.float32)
        self.A  = np.zeros(cap, dtype=np.int64)
        self.R  = np.zeros(cap, dtype=np.float32)
        self.D  = np.zeros(cap, dtype=np.float32)
        self.P  = np.ones(cap,  dtype=np.float64)

    def push(self, s, a, r, sn, d):
        mp = self.P[:max(self.size, 1)].max()
        i  = self.ptr
        self.S[i]=s; self.Sn[i]=sn; self.A[i]=a
        self.R[i]=r; self.D[i]=d;   self.P[i]=mp if mp>0 else 1.0
        self.ptr  = (self.ptr+1) % self.cap
        self.size = min(self.size+1, self.cap)

    def sample(self, bs):
        vp   = self.P[:self.size] ** self.alpha
        prob = vp / vp.sum()
        idx  = np.random.choice(self.size, size=bs,
                                replace=(self.size < bs), p=prob)
        isw  = (1.0 / (self.size * prob[idx])) ** self.beta
        isw  = (isw / isw.max()).astype(np.float32)
        t = lambda a: torch.tensor(a, device=DEVICE)
        return t(self.S[idx]),t(self.A[idx]),t(self.R[idx]),\
               t(self.Sn[idx]),t(self.D[idx]),idx,t(isw)

    def update_priorities(self, idx, td):
        self.P[idx] = np.abs(td) + 1e-6


# ─────────────────────────────────────────────────────────────
#  7.  TRAINING STEP
# ─────────────────────────────────────────────────────────────
def train_step(online, target, opt, buf):
    S,A,R,Sn,D,idx,isw = buf.sample(BATCH_SIZE)
    online.train(); target.eval()
    q_pred = online(S).gather(1, A.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        best_a = online(Sn).argmax(dim=1, keepdim=True)
        q_next = target(Sn).gather(1, best_a).squeeze(1)
        q_tgt  = R + GAMMA * q_next * (1.0 - D)
    td_err = q_pred - q_tgt
    loss   = (isw * F.huber_loss(q_pred, q_tgt, reduction="none")).mean()
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(online.parameters(), GRAD_CLIP)
    opt.step()
    buf.update_priorities(idx, td_err.detach().cpu().numpy())
    return float(td_err.abs().mean().item())


# ─────────────────────────────────────────────────────────────
#  8.  HELPERS
# ─────────────────────────────────────────────────────────────
def cosine_eps(ep, n):
    return EPS_END + 0.5*(EPS_START-EPS_END)*(1+math.cos(math.pi*ep/max(n-1,1)))

def per_snr(snr_a, val_a, su):
    return np.array([val_a[snr_a==sv].mean() if (snr_a==sv).any() else np.nan
                     for sv in su])

def conf_matrix(true_l, pred_l, n=N_ACTIONS):
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(true_l, pred_l): cm[t, p] += 1
    return cm

def savefig(fig, name):
    p = os.path.join(SAVE_DIR, name)
    fig.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  ✓  {p}"); plt.close(fig)

def smooth(v, w=None):
    if w is None: w = max(1, len(v)//15)
    w = max(1, min(w, len(v)))
    return np.convolve(v, np.ones(w)/w, mode="same")

def agrid(ax, both=False):
    ax.grid(True, which="both" if both else "major",
            alpha=0.3, color=COLORS["grid"])


# ─────────────────────────────────────────────────────────────
#  9.  TRAINING LOOP  (FIX 3: load checkpoint AFTER loop)
# ─────────────────────────────────────────────────────────────
def train(env, online, target, opt, sched, buf):
    print(f"\n{'─'*62}")
    print(f"  RL Training | {N_EPISODES} ep | {EPISODE_LEN} steps/ep")
    print(f"  Batch {BATCH_SIZE} | Buffer {BUF_CAPACITY:,} | Warm {WARM_START}")
    print(f"{'─'*62}")

    logs = dict(reward=[], acc=[], td=[], eps=[])
    best_acc = 0.0

    for ep in range(N_EPISODES):
        eps      = cosine_eps(ep, N_EPISODES)
        buf.beta = min(1.0, PER_BETA_0+(1-PER_BETA_0)*ep/max(N_EPISODES-1,1))

        state = env.reset(); done = False
        ep_r = ep_acc = ep_td = 0.0; steps = 0

        while not done:
            action, _ = online.act_epsilon_greedy(state, eps)
            next_s, reward, done, info = env.step(action)
            buf.push(state, action, reward, next_s, float(done))
            ep_r   += reward
            ep_acc += float(action == info["optimal"])
            state   = next_s; steps += 1

            if buf.size >= WARM_START:
                td = train_step(online, target, opt, buf)
                with torch.no_grad():
                    for po, pt in zip(online.parameters(), target.parameters()):
                        pt.data.copy_(TAU*po.data + (1-TAU)*pt.data)
                ep_td += td

        sched.step()   # once per episode
        if ep % HARD_UPD_EP == 0:
            target.load_state_dict(online.state_dict())

        # FIX 3: record logs BEFORE any checkpoint load
        logs["reward"].append(ep_r  / steps)
        logs["acc"].append(ep_acc / steps * 100)
        logs["td"].append(ep_td  / max(steps, 1))
        logs["eps"].append(eps)

        if logs["acc"][-1] > best_acc:
            best_acc = logs["acc"][-1]
            torch.save(online.state_dict(), "dqn_best.pt")

        if ep % 25 == 0 or ep == N_EPISODES-1:
            print(f"  Ep {ep+1:4d}/{N_EPISODES} | "
                  f"reward={logs['reward'][-1]:+7.3f} | "
                  f"acc={logs['acc'][-1]:5.1f}% | "
                  f"|TD|={logs['td'][-1]:.4f} | "
                  f"ε={eps:.3f} | best={best_acc:.1f}%")

    # FIX 3: load best checkpoint AFTER loop (not inside, no log entry)
    if os.path.exists("dqn_best.pt"):
        online.load_state_dict(torch.load("dqn_best.pt", map_location=DEVICE))
        print(f"\n  Loaded best checkpoint  (acc={best_acc:.1f}%)")
    print(f"{'─'*62}\n")
    return logs


# ─────────────────────────────────────────────────────────────
#  10.  GREEDY EVALUATION  (full sorted dataset)
# ─────────────────────────────────────────────────────────────
def evaluate(X_scl, y, df_raw, online):
    print("  Greedy evaluation on full dataset (sorted by SNR) …")
    order = df_raw.sort_values("SNR_dB").index
    Xs = X_scl[order]; ys = y[order]
    dfs = df_raw.iloc[order].reset_index(drop=True)

    records = []; online.eval(); prev = 0
    with torch.no_grad():
        for i in range(len(Xs)):
            s   = torch.tensor(Xs[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
            q   = online(s).squeeze(0).cpu().numpy()
            act = int(np.argmax(q))
            row = dfs.iloc[i]; opt = int(ys[i])
            (_, ber_sel, tput_sel, lat_sel,
             r_qual, r_lat, r_exp, r_sw) = compute_reward(row, act, opt, prev)
            records.append(dict(
                snr=float(row["SNR_dB"]), action=act, optimal=opt,
                ber_vlc=max(float(row["BER_VLC"]),1e-9),
                ber_wifi=max(float(row["BER_WiFi"]),1e-9),
                ber_mrc=max(float(row["BER_MRC"]),1e-9),
                ber_sel=ber_sel, tput=tput_sel, lat=lat_sel,
                r_qual=r_qual, r_lat=r_lat, r_expert=r_exp, r_switch=r_sw,
            ))
            prev = act

    ev  = pd.DataFrame(records)
    acc = (ev["action"]==ev["optimal"]).mean()*100
    dist = {ACTION_NAMES[i]: int((ev["action"]==i).sum()) for i in range(N_ACTIONS)}
    print(f"  Accuracy vs optimal label : {acc:.2f}%")
    print(f"  Action distribution       : {dist}\n")
    return ev   # FIX 1: single ev used by ALL plots


# ─────────────────────────────────────────────────────────────
#  11.  PLOTS  (FIX 1: all receive the SAME ev)
# ─────────────────────────────────────────────────────────────

def plot_ber(ev):
    su = sorted(ev["snr"].unique()); sa = ev["snr"].values
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for col, lbl, c, mk, lw in [
        ("ber_vlc",  "VLC (fixed)",  COLORS["VLC"],  "s", 1.4),
        ("ber_wifi", "WiFi (fixed)", COLORS["WiFi"], "o", 1.4),
        ("ber_mrc",  "MRC (fixed)",  COLORS["MRC"],  "D", 1.4),
        ("ber_sel",  "DQN select",   COLORS["DQN"],  "^", 2.8),
    ]:
        v = per_snr(sa, ev[col].values, su)
        ax.semilogy(su, v, f"{mk}-", lw=lw, ms=5 if lw>2 else 4, color=c,
                    label=lbl, zorder=5 if lw>2 else 3,
                    alpha=1.0 if lw>2 else 0.75)
    ax.set(xlabel="SNR (dB)", ylabel="BER",
           title="Fig 1 — BER vs SNR: DQN vs Fixed Strategies", ylim=(1e-8, 1.5))
    ax.legend(loc="upper right", fontsize=9); agrid(ax, both=True)
    ax.yaxis.set_major_formatter(mticker.LogFormatterMathtext())
    plt.tight_layout(); savefig(fig, "fig1_ber_vs_snr.png")


def plot_throughput(ev, df_raw):
    su = sorted(ev["snr"].unique()); dfs = df_raw.sort_values("SNR_dB")
    tp_v = per_snr(dfs["SNR_dB"].values, dfs["Throughput_VLC_Mbps"].values,  su)
    tp_w = per_snr(dfs["SNR_dB"].values, dfs["Throughput_WiFi_Mbps"].values, su)
    tp_d = per_snr(ev["snr"].values, ev["tput"].values, su)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(su,tp_v,"s--",lw=1.4,ms=4,color=COLORS["VLC"],label="VLC (fixed)",alpha=0.75)
    ax.plot(su,tp_w,"o--",lw=1.4,ms=4,color=COLORS["WiFi"],label="WiFi (fixed)",alpha=0.75)
    ax.plot(su,tp_d,"^-",lw=2.8,ms=6,color=COLORS["DQN"],label="DQN select",zorder=5)
    ax.fill_between(su,tp_w,tp_d,where=(tp_d>=tp_w),alpha=0.12,
                    color=COLORS["DQN"],label="DQN gain vs WiFi")
    ax.set(xlabel="SNR (dB)",ylabel="Throughput (Mbps)",title="Fig 2 — Throughput vs SNR")
    ax.legend(fontsize=9); agrid(ax); plt.tight_layout(); savefig(fig,"fig2_throughput.png")


def plot_latency(ev, df_raw):
    su = sorted(ev["snr"].unique()); dfs = df_raw.sort_values("SNR_dB")
    lt_v = per_snr(dfs["SNR_dB"].values, dfs["Latency_VLC_ms"].values,  su)
    lt_w = per_snr(dfs["SNR_dB"].values, dfs["Latency_WiFi_ms"].values, su)
    lt_d = per_snr(ev["snr"].values, ev["lat"].values, su)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(su,lt_v,"s--",lw=1.4,ms=4,color=COLORS["VLC"],label="VLC (fixed)",alpha=0.75)
    ax.plot(su,lt_w,"o--",lw=1.4,ms=4,color=COLORS["WiFi"],label="WiFi (fixed)",alpha=0.75)
    ax.plot(su,lt_d,"^-",lw=2.8,ms=6,color=COLORS["DQN"],label="DQN select",zorder=5)
    ax.set(xlabel="SNR (dB)",ylabel="Latency (ms)",title="Fig 3 — Latency vs SNR")
    ax.legend(fontsize=9); agrid(ax); plt.tight_layout(); savefig(fig,"fig3_latency.png")


def plot_actions(ev):
    su = sorted(ev["snr"].unique()); sa = ev["snr"].values; acts = ev["action"].values
    freq = np.zeros((N_ACTIONS, len(su)))
    for j, sv in enumerate(su):
        m = sa==sv; tot = m.sum()
        for a in range(N_ACTIONS): freq[a,j] = (acts[m]==a).sum()/tot*100
    fig, ax = plt.subplots(figsize=(13, 4.5))
    bot = np.zeros(len(su))
    for a in range(N_ACTIONS):
        c = [COLORS["VLC"],COLORS["WiFi"],COLORS["MRC"]][a]
        ax.bar(su, freq[a], bottom=bot, color=c, width=0.75,
               label=ACTION_NAMES[a], alpha=0.9); bot += freq[a]
    ax.set(xlabel="SNR (dB)", ylabel="Selection (%)",
           title="Fig 4 — DQN Action Distribution per SNR",
           xlim=(-1,max(su)+1), ylim=(0,105))
    ax.legend(loc="upper right", fontsize=9); agrid(ax)
    plt.tight_layout(); savefig(fig, "fig4_action_dist.png")


def plot_reward(logs):
    n = len(logs["reward"]); ex = np.arange(1, n+1)
    fig, ax1 = plt.subplots(figsize=(11, 4.5)); ax2 = ax1.twinx()
    ax1.plot(ex, logs["reward"], lw=0.6, color=COLORS["DQN"], alpha=0.25)
    ax1.plot(ex, smooth(logs["reward"]), lw=2.4, color=COLORS["DQN"], label="Reward (smoothed)")
    ax2.plot(ex, logs["eps"], lw=1.5, ls="--", color="#999", label="ε")
    ax1.set(xlabel="Episode", ylabel="Avg reward / step",
            title="Fig 5 — Training Reward & Exploration Rate")
    ax2.set_ylabel("ε", color="#888"); ax2.tick_params(axis="y", colors="#888")
    agrid(ax1)
    l1,lb1=ax1.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, fontsize=9, loc="upper right")
    plt.tight_layout(); savefig(fig, "fig5_reward_curve.png")


def plot_accuracy(logs):
    n = len(logs["acc"]); ex = np.arange(1, n+1)
    fig, ax1 = plt.subplots(figsize=(11, 4.5)); ax2 = ax1.twinx()
    ax1.plot(ex, logs["acc"], lw=0.5, color="#4DAEEF", alpha=0.25)
    ax1.plot(ex, smooth(logs["acc"]), lw=2.4, color="#4DAEEF", label="Accuracy (smoothed)")
    ax1.axhline(100/N_ACTIONS, ls=":", color="gray", lw=1.2, label="Random baseline")
    ax2.plot(ex, logs["td"], lw=0.8, color="#FF6B6B", alpha=0.5, label="|TD error|")
    ax2.plot(ex, smooth(logs["td"]), lw=2.0, color="#CC2222")
    ax2.set_ylabel("Mean |TD error|", color="#CC2222")
    ax2.tick_params(axis="y", colors="#CC2222")
    ax1.set(xlabel="Episode", ylabel="Accuracy vs optimal (%)",
            title="Fig 6 — Training Accuracy & TD Error", ylim=(0, 108))
    agrid(ax1)
    l1,lb1=ax1.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, fontsize=9, loc="lower right")
    plt.tight_layout(); savefig(fig, "fig6_accuracy_td.png")


def plot_confusion(ev):
    """FIX 6: Single panel with both count and row-% annotations."""
    cm   = conf_matrix(ev["optimal"].values, ev["action"].values)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=100)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Row %")

    for r in range(N_ACTIONS):
        for c in range(N_ACTIONS):
            dark = cm_n[r, c] > 50
            ax.text(c, r-0.15, f"{cm[r,c]:,}",
                    ha="center", va="center", fontsize=11, fontweight="bold",
                    color="white" if dark else "black")
            ax.text(c, r+0.20, f"({cm_n[r,c]:.1f}%)",
                    ha="center", va="center", fontsize=9,
                    color="white" if dark else "#444")

    ax.set(xticks=range(N_ACTIONS), yticks=range(N_ACTIONS),
           xticklabels=ACTION_NAMES, yticklabels=ACTION_NAMES,
           xlabel="DQN Predicted", ylabel="Optimal (ground truth)",
           title="Fig 7 — Confusion Matrix  (count  &  row %)")
    plt.tight_layout(); savefig(fig, "fig7_confusion.png")


def plot_reward_breakdown(ev):
    su  = sorted(ev["snr"].unique()); sa = ev["snr"].values
    comps = ["r_qual","r_lat","r_expert"]
    lbls  = ["Quality (BER+Tput+Rel)","Latency","Expert signal"]
    clrs  = ["#2196F3","#FF9800","#4CAF50"]
    fig, ax = plt.subplots(figsize=(13, 5))
    bot = np.zeros(len(su))
    for comp, lbl, c in zip(comps, lbls, clrs):
        v = per_snr(sa, ev[comp].values, su)
        ax.bar(su, np.maximum(v,0), bottom=bot, color=c, width=0.7, label=lbl, alpha=0.88)
        bot += np.maximum(v, 0)
    sw = -per_snr(sa, ev["r_switch"].values, su)
    ax.bar(su, sw, color="#F44336", width=0.7, label="Switch penalty (−)", alpha=0.88)
    ax.axhline(0, color="black", lw=0.8)
    ax.set(xlabel="SNR (dB)", ylabel="Avg reward component",
           title="Fig 8 — Reward Component Breakdown per SNR")
    ax.legend(fontsize=9, ncol=2, loc="upper left"); agrid(ax)
    plt.tight_layout(); savefig(fig, "fig8_reward_breakdown.png")


# ─────────────────────────────────────────────────────────────
#  12.  DASHBOARD  (FIX 1 & 2: same ev, identical BER panel)
# ─────────────────────────────────────────────────────────────
def plot_dashboard(ev, logs, df_raw):
    """
    FIX 1: receives ev from evaluate() — identical data as standalone figs.
    FIX 2: BER panel [0,0] has same 4 lines as fig1.
    Q-value heatmap replaced with per-SNR accuracy bar chart.
    """
    su  = sorted(ev["snr"].unique()); sa = ev["snr"].values
    n   = len(logs["reward"]); ex = np.arange(1, n+1)
    dfs = df_raw.sort_values("SNR_dB")
    acc_final = (ev["action"]==ev["optimal"]).mean()*100

    fig, axes = plt.subplots(3, 3, figsize=(19, 14))
    fig.suptitle(
        f"WiFi+VLC  Dueling Double DQN + PER  —  Summary Dashboard\n"
        f"20 500 samples · {n} episodes · {EPISODE_LEN} steps/ep · "
        f"greedy acc = {acc_final:.1f}%",
        fontsize=13, fontweight="bold")

    # ── [0,0] BER  (FIX 2: identical 4-line plot) ─────────────────────
    ax = axes[0,0]
    for col,lbl,c,mk,lw in [
        ("ber_vlc","VLC",COLORS["VLC"],"s",1.2),
        ("ber_wifi","WiFi",COLORS["WiFi"],"o",1.2),
        ("ber_mrc","MRC",COLORS["MRC"],"D",1.2),         # FIX 2: added MRC
        ("ber_sel","DQN",COLORS["DQN"],"^",2.2),
    ]:
        v = per_snr(sa, ev[col].values, su)
        ax.semilogy(su,v,f"{mk}-",lw=lw,ms=4,color=c,label=lbl,
                    alpha=1 if lw>2 else 0.7)
    ax.set(title="BER vs SNR",xlabel="SNR (dB)",ylabel="BER")
    ax.legend(fontsize=7); agrid(ax,both=True)
    ax.yaxis.set_major_formatter(mticker.LogFormatterMathtext())

    # ── [0,1] Throughput ───────────────────────────────────────────────
    ax = axes[0,1]
    tp_v=per_snr(dfs["SNR_dB"].values,dfs["Throughput_VLC_Mbps"].values,su)
    tp_w=per_snr(dfs["SNR_dB"].values,dfs["Throughput_WiFi_Mbps"].values,su)
    tp_d=per_snr(sa,ev["tput"].values,su)
    ax.plot(su,tp_v,"s--",lw=1.2,ms=3,color=COLORS["VLC"],label="VLC",alpha=0.7)
    ax.plot(su,tp_w,"o--",lw=1.2,ms=3,color=COLORS["WiFi"],label="WiFi",alpha=0.7)
    ax.plot(su,tp_d,"^-",lw=2.2,ms=4,color=COLORS["DQN"],label="DQN")
    ax.set(title="Throughput vs SNR",xlabel="SNR (dB)",ylabel="Mbps")
    ax.legend(fontsize=7); agrid(ax)

    # ── [0,2] Latency ──────────────────────────────────────────────────
    ax = axes[0,2]
    lt_v=per_snr(dfs["SNR_dB"].values,dfs["Latency_VLC_ms"].values,su)
    lt_w=per_snr(dfs["SNR_dB"].values,dfs["Latency_WiFi_ms"].values,su)
    lt_d=per_snr(sa,ev["lat"].values,su)
    ax.plot(su,lt_v,"s--",lw=1.2,ms=3,color=COLORS["VLC"],label="VLC",alpha=0.7)
    ax.plot(su,lt_w,"o--",lw=1.2,ms=3,color=COLORS["WiFi"],label="WiFi",alpha=0.7)
    ax.plot(su,lt_d,"^-",lw=2.2,ms=4,color=COLORS["DQN"],label="DQN")
    ax.set(title="Latency vs SNR",xlabel="SNR (dB)",ylabel="ms")
    ax.legend(fontsize=7); agrid(ax)

    # ── [1,0] Action distribution ──────────────────────────────────────
    ax = axes[1,0]
    freq=np.zeros((N_ACTIONS,len(su))); bot=np.zeros(len(su))
    for j,sv in enumerate(su):
        m=sa==sv; tot=m.sum()
        for a in range(N_ACTIONS): freq[a,j]=(ev["action"].values[m]==a).sum()/tot*100
    for a in range(N_ACTIONS):
        c=[COLORS["VLC"],COLORS["WiFi"],COLORS["MRC"]][a]
        ax.bar(su,freq[a],bottom=bot,color=c,width=0.75,label=ACTION_NAMES[a],alpha=0.9)
        bot+=freq[a]
    ax.set(title="Action Distribution",xlabel="SNR (dB)",ylabel="%")
    ax.legend(fontsize=7); agrid(ax)

    # ── [1,1] Training reward ──────────────────────────────────────────
    ax = axes[1,1]
    ax.plot(ex,logs["reward"],lw=0.5,color=COLORS["DQN"],alpha=0.25)
    ax.plot(ex,smooth(logs["reward"]),lw=2.2,color=COLORS["DQN"])
    ax.set(title="Training Reward",xlabel="Episode",ylabel="Avg reward/step"); agrid(ax)

    # ── [1,2] Training accuracy ────────────────────────────────────────
    ax = axes[1,2]
    ax.plot(ex,logs["acc"],lw=0.5,color="#4DAEEF",alpha=0.25)
    ax.plot(ex,smooth(logs["acc"]),lw=2.2,color="#4DAEEF")
    ax.axhline(100/N_ACTIONS,ls=":",color="gray",lw=1.2)
    ax.set(title="Training Accuracy (%)",xlabel="Episode",
           ylabel="Acc (%)",ylim=(0,108)); agrid(ax)

    # ── [2,0] Confusion matrix  (FIX 1 & 6: same ev, single panel) ────
    ax = axes[2,0]
    cm   = conf_matrix(ev["optimal"].values, ev["action"].values)   # FIX 1
    cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
    im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=100)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for r in range(N_ACTIONS):
        for c in range(N_ACTIONS):
            dark = cm_n[r,c] > 50
            ax.text(c, r-0.15, f"{cm[r,c]:,}", ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color="white" if dark else "black")
            ax.text(c, r+0.20, f"({cm_n[r,c]:.0f}%)", ha="center", va="center",
                    fontsize=8, color="white" if dark else "#444")
    ax.set(title="Confusion Matrix (count & row %)",
           xticks=range(N_ACTIONS), yticks=range(N_ACTIONS),
           xticklabels=ACTION_NAMES, yticklabels=ACTION_NAMES,
           xlabel="Predicted", ylabel="True")

    # ── [2,1] Per-SNR accuracy  (replaces Q-heatmap — FIX 5) ──────────
    ax = axes[2,1]
    snr_acc = []
    for sv in su:
        m = sa == sv
        snr_acc.append((ev["action"].values[m]==ev["optimal"].values[m]).mean()*100)
    bars = ax.bar(su, snr_acc, width=0.75, color=COLORS["DQN"], alpha=0.85)
    ax.axhline(100/N_ACTIONS, ls=":", color="gray", lw=1.2, label="Random baseline")
    ax.set(title="Per-SNR Accuracy (%)", xlabel="SNR (dB)",
           ylabel="Accuracy (%)", ylim=(0, 108))
    ax.legend(fontsize=7); agrid(ax)

    # ── [2,2] Reward breakdown ─────────────────────────────────────────
    ax = axes[2,2]
    comps=["r_qual","r_lat","r_expert"]; lbls=["Quality","Latency","Expert"]
    clrs=["#2196F3","#FF9800","#4CAF50"]; bot2=np.zeros(len(su))
    for comp,lbl,c in zip(comps,lbls,clrs):
        v=per_snr(sa,ev[comp].values,su)
        ax.bar(su,np.maximum(v,0),bottom=bot2,color=c,width=0.7,label=lbl,alpha=0.88)
        bot2+=np.maximum(v,0)
    sw=-per_snr(sa,ev["r_switch"].values,su)
    ax.bar(su,sw,color="#F44336",width=0.7,label="Switch−",alpha=0.88)
    ax.axhline(0,color="black",lw=0.6)
    ax.set(title="Reward Breakdown",xlabel="SNR (dB)",ylabel="Avg component")
    ax.legend(fontsize=7,ncol=2); agrid(ax)

    plt.tight_layout(); savefig(fig, "fig0_dashboard.png")


# ─────────────────────────────────────────────────────────────
#  13.  MAIN
# ─────────────────────────────────────────────────────────────
def main():
    X_scl, X_raw, y, scaler, df_raw = load_dataset(CSV_PATH)
    env = LinkEnv(X_scl, y, df_raw, ep_len=EPISODE_LEN)

    online = DuelingDQN(N_FEATURES, N_ACTIONS).to(DEVICE)
    target = DuelingDQN(N_FEATURES, N_ACTIONS).to(DEVICE)
    target.load_state_dict(online.state_dict()); target.eval()

    total_params = sum(p.numel() for p in online.parameters())
    print(f"\n  Network params : {total_params:,}")

    # FIX 4: behavioral cloning warm-start
    pretrain_bc(online, X_scl, y, epochs=BC_EPOCHS)
    target.load_state_dict(online.state_dict())   # sync target after BC

    opt   = optim.AdamW(online.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPISODES, eta_min=LR*0.05)
    buf   = PERBuffer(BUF_CAPACITY, N_FEATURES)

    logs = train(env, online, target, opt, sched, buf)

    # FIX 1: single ev used by ALL plots
    ev = evaluate(X_scl, y, df_raw, online)

    path = os.path.join(SAVE_DIR, "dqn_model.pt")
    torch.save({"model_state": online.state_dict(),
                "scaler_mean": scaler.mean_, "scaler_std": scaler.scale_,
                "n_features": N_FEATURES, "n_actions": N_ACTIONS,
                "action_names": ACTION_NAMES}, path)
    print(f"  ✓  Model saved → {path}\n")

    print("  Generating figures …")
    df_s = df_raw.sort_values("SNR_dB").reset_index(drop=True)
    plot_ber(ev)
    plot_throughput(ev, df_s)
    plot_latency(ev, df_s)
    plot_actions(ev)
    plot_reward(logs)
    plot_accuracy(logs)
    plot_confusion(ev)           # FIX 6: single clean panel
    plot_reward_breakdown(ev)
    plot_dashboard(ev, logs, df_s)   # FIX 1 & 2: same ev, 4-line BER

    print("\n  ✓  9 figures saved  (Q-heatmap removed):")
    for f in ["fig0_dashboard.png  ← 3×3 summary",
              "fig1_ber_vs_snr.png","fig2_throughput.png",
              "fig3_latency.png","fig4_action_dist.png",
              "fig5_reward_curve.png","fig6_accuracy_td.png",
              "fig7_confusion.png","fig8_reward_breakdown.png"]:
        print(f"     {f}")


if __name__ == "__main__":
    main()

  Dataset   : AI_switching_dataset_NEW.csv
  Rows      : 20,500   Features : 14
  SNR range : 0–40 dB (41 unique, 500 rows each)
  Action 0 (VLC ): 10,307  (50.3%)
  Action 1 (WiFi):  2,508  (12.2%)
  Action 2 (MRC ):  7,685  (37.5%)
  Device    : cpu

  Network params : 470,532

  Behavioral cloning pre-training (20 epochs) …
    BC ep   1/20  loss=0.1500  acc=94.2%
    BC ep   6/20  loss=0.0448  acc=98.3%
    BC ep  11/20  loss=0.0443  acc=98.3%
    BC ep  16/20  loss=0.0446  acc=98.5%
    BC ep  20/20  loss=0.0306  acc=98.9%


──────────────────────────────────────────────────────────────
  RL Training | 300 ep | 500 steps/ep
  Batch 256 | Buffer 50,000 | Warm 512
──────────────────────────────────────────────────────────────
  Ep    1/300 | reward= +2.270 | acc= 33.6% | |TD|=0.0000 | ε=1.000 | best=33.6%
  Ep   26/300 | reward= +2.403 | acc= 35.6% | |TD|=4.9383 | ε=0.983 | best=36.8%
  Ep   51/300 | reward= +2.440 | acc= 37.0% | |TD|=6.8469 | ε=0.933 | best=40.4%
  Ep   76/300 | re